# Camada BRONZE — Ingestão Bruta
Lê os CSVs e persiste como Delta Lake sem nenhuma transformação.

In [5]:
pip install faker

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
import sys
sys.path.insert(0, '/opt/spark/work-dir')
from tools.spark_session import get_spark

spark = get_spark('Bronze - Ingestão')
spark.sparkContext.setLogLevel('WARN')
print('Spark iniciado:', spark.version)

Spark iniciado: 4.0.0


26/06/18 23:16:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
# Gera os datasets de exemplo (roda só uma vez)
import subprocess, sys
subprocess.run([sys.executable, '/opt/spark/work-dir/datasets/gerar_dados.py'], check=True)

Gerando fornecedores.csv  (50 linhas)...
Gerando clientes.csv  (100,000 linhas)...
Gerando cupons.csv  (200 linhas)...
Gerando produtos.csv  (1,000 linhas)...
Gerando vendas.csv, fretes.csv, pagamentos.csv  (1,000,000 linhas)...
Gerando avaliacoes.csv  (420,408 linhas)...

Datasets gerados:
  avaliacoes.csv           21.5 MB
  clientes.csv              7.5 MB
  cupons.csv                0.0 MB
  fornecedores.csv          0.0 MB
  fretes.csv               40.7 MB
  pagamentos.csv           38.5 MB
  produtos.csv              0.0 MB
  vendas.csv               58.7 MB


CompletedProcess(args=['/usr/bin/python3', '/opt/spark/work-dir/datasets/gerar_dados.py'], returncode=0)

In [9]:
DATASETS = '/opt/spark/work-dir/datasets'
BRONZE   = '/opt/spark/work-dir/warehouse/bronze'

for table in ['clientes', 'produtos', 'vendas','fretes', 'pagamentos','cupons', 'avaliacoes']:
    df = (
        spark.read
        .option('header', True)
        .option('inferSchema', True)
        .csv(f'{DATASETS}/{table}.csv')
    )
    df.write.format('delta').mode('overwrite').save(f'{BRONZE}/{table}')
    print(f'[bronze/{table}] {df.count()} linhas salvas')

26/06/18 23:19:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[bronze/clientes] 100000 linhas salvas
[bronze/produtos] 1000 linhas salvas


26/06/18 23:19:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/18 23:19:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/18 23:19:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/18 23:19:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/06/18 23:19:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/06/18 23:19:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/06/18 23:19:47 WARN MemoryManager: Total allocation exceeds 95.

[bronze/vendas] 1000000 linhas salvas


26/06/18 23:19:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/18 23:19:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/18 23:19:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/18 23:19:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/06/18 23:19:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/18 23:19:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/18 23:19:53 WARN MemoryManager: Total allocation exceeds 95.0

[bronze/fretes] 1000000 linhas salvas


26/06/18 23:19:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/18 23:19:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/18 23:19:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/18 23:19:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/18 23:19:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


[bronze/pagamentos] 1000000 linhas salvas
[bronze/cupons] 200 linhas salvas
[bronze/avaliacoes] 420408 linhas salvas


In [12]:
# Inspeciona o schema de cada tabela
for table in ['clientes', 'produtos', 'vendas','fretes', 'pagamentos','cupons', 'avaliacoes']:
    df = spark.read.format('delta').load(f'{BRONZE}/{table}')
    print(f'\n=== {table.upper()} ===')
    df.printSchema()
    df.show(5, truncate=False)


=== CLIENTES ===
root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- tipo_cliente: string (nullable = true)
 |-- documento: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- data_cadastro: date (nullable = true)
 |-- ativo: integer (nullable = true)
 |-- score_credito: double (nullable = true)

+----------+------------------+------------+--------------+--------+--------------+-------------+-----+-------------+
|cliente_id|nome              |tipo_cliente|documento     |regiao  |cidade        |data_cadastro|ativo|score_credito|
+----------+------------------+------------+--------------+--------+--------------+-------------+-----+-------------+
|53394     |Théo Cirino       |PF          |460.529.387-65|Sudeste |Rio de Janeiro|2023-08-10   |1    |350.8        |
|53395     |Vinicius Pinto    |PF          |134.568.907-10|Norte   |Manaus        |2022-02-06   |0    |925.0        |
|53396     |Maria Cl

In [14]:
# Histórico de versões Delta
from delta.tables import DeltaTable
for table in ['clientes', 'produtos', 'vendas','fretes', 'pagamentos','cupons', 'avaliacoes']:
    dt   = DeltaTable.forPath(spark, f'{BRONZE}/{table}')
    hist = dt.history(1).select('version', 'timestamp', 'operation').collect()[0]
    print(f'[{table}] versão={hist["version"]}  op={hist["operation"]}')

[clientes] versão=1  op=WRITE
[produtos] versão=1  op=WRITE
[vendas] versão=1  op=WRITE
[fretes] versão=0  op=WRITE
[pagamentos] versão=0  op=WRITE
[cupons] versão=0  op=WRITE
[avaliacoes] versão=0  op=WRITE
